# Pipeline SFT — FOL (NL → premises-FOL)

`src/models/fol_model/` + `src/data/fol_dataset.py`. Dữ liệu: `premises_nl` / `premises_fol` (bỏ mẫu lệch độ dài). Hub: prefix **`fol-`**.

- Có thể trỏ `FOL_SFT_PROCESSED_DIR` tới `data/processed/fol_sft` (export lọc) hoặc dùng chung `logic_sft`.
- Sau train: `fol_eval_*` trong `experiment_log.json` (exact match JSON).
- Notebook: dependency + `%run -i` các stage.

## 0. Dependency (chạy khi cần)

In [ ]:
# %pip install -q transformers datasets peft "trl>=0.16.0" accelerate bitsandbytes scikit-learn huggingface_hub gdown python-dotenv
# %pip install -q trl

## 1. Secrets (Kaggle) / token HF

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    HF_Token = UserSecretsClient().get_secret("HF_Token")
except Exception:
    HF_Token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""

## 2. Bootstrap path cho `services` + `fol_model`

In [ ]:
%run -i project_bootstrap.py

## 3. Cấu hình FOL — chỉnh tại đây

In [ ]:
from services.config_fol import FolSFTConfig

cfg = FolSFTConfig.from_env()
print("Config OK", cfg.model_name)
print("HF repo (push):", cfg.resolved_hf_repo())

## 3b. (Tuỳ chọn) Export `fol_sft/*.csv` đã lọc
Chạy trong một ô riêng nếu muốn tách khỏi `logic_sft`:

```python
from pathlib import Path
from data.fol_dataset import export_filtered_fol_csvs

root = Path(cfg.app_dir).resolve()
export_filtered_fol_csvs(root / "data/processed/logic_sft", root / "data/processed/fol_sft")
```

## 4. Tải dữ liệu thô (Drive) — theo `LOGIC_DATA_SOURCE`
Bỏ qua nếu đã có JSON trong `data/raw/`.

In [ ]:
from services.config import LogicSFTConfig

if LogicSFTConfig.from_env().data_source == "drive":
    get_ipython().run_line_magic("run", "-i fol_model_stage_drive.py")
else:
    print("DATA_SOURCE=local — bỏ qua tải Drive.")

## 5. Fine-tune FOL (LoRA SFT)

In [ ]:
%run -i fol_model_stage_train.py

## 6. Push Hugging Face (tùy chọn)
Đặt `FOL_PUSH_TO_HUB=true` trong `.env` và có `HF_Token`.

In [ ]:
%run -i fol_model_stage_push.py